# Gold Processing — Date

Enriches `ecommerce.silver.silver_date` with three derived columns into `ecommerce.gold.gold_date`:

* **`date_id`** — the date without hyphens as an integer surrogate key (e.g. `2025-08-01` → `20250801`)
* **`is_weekend`** — boolean: `true` when `day_name` is Saturday or Sunday, `false` for weekdays
* **`month_name`** — full month name from the date (e.g. `August`)

All `silver_date` columns (`date`, `year`, `day_name`, `quarter`, `week_of_year`) and metadata (`source_file`, `ingested_at`) pass through.

In [0]:
from pyspark.sql.functions import col, date_format

target_table = "ecommerce.gold.gold_date"

df_gold_date = (
    spark.table("ecommerce.silver.silver_date")
    .withColumn("date_id", date_format(col("date"), "yyyyMMdd").cast("int"))  # date without '-'
    .withColumn("is_weekend", col("day_name").isin("Saturday", "Sunday"))       # Sat/Sun = true
    .withColumn("month_name", date_format(col("date"), "MMMM"))                 # full month name
    .select(
        "date_id",
        "date",
        "year",
        "month_name",
        "day_name",
        "is_weekend",
        "quarter",
        "week_of_year",
        "source_file",
        "ingested_at",
    )
)

# Idempotent full load into the gold table
(
    df_gold_date.write
    .mode("overwrite")
    .saveAsTable(target_table)
)

print(f"Wrote {spark.table(target_table).count()} rows to {target_table}")

In [0]:
# Verify the persisted gold table
spark.sql("""
    SELECT
      COUNT(*) AS rows,
      COUNT_IF(date_id IS NULL) AS null_date_ids,
      COUNT_IF(is_weekend != (day_name IN ('Saturday', 'Sunday'))) AS weekend_mismatches,
      COUNT_IF(month_name != date_format(date, 'MMMM')) AS month_mismatches,
      SUM(CAST(is_weekend AS INT)) AS weekend_days
    FROM ecommerce.gold.gold_date
""").show()

spark.sql("""
    SELECT month_name, COUNT(*) AS days, SUM(CAST(is_weekend AS INT)) AS weekend_days
    FROM ecommerce.gold.gold_date
    GROUP BY month_name
    ORDER BY min(date)
""").show(truncate=False)

display(spark.table("ecommerce.gold.gold_date").limit(5))